# Ch12. Text Analysis and Generation

In [3]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

Downloaded thinkpython.py
Downloaded diagram.py


In [4]:
download('https://www.gutenberg.org/cache/epub/43/pg43.txt');

Downloaded pg43.txt


## Unique Words

In [2]:
# Clean up the book
def is_special_line(line):
    return line.strip().startswith('*** ')

def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w')

    # Omit the header text
    for line in reader:
        if is_special_line(line):
            break
    
    # Read up the footer text
    for line in reader:
        if is_special_line(line):
            break;
        writer.write(line)

    reader.close()
    writer.close()
    
filename = 'dr_jekyll.txt'
clean_file('pg43.txt', filename)

In [3]:
# Count the number of unique words
unique_words = {}
for line in open(filename):
    seq = line.split()
    for word in seq: 
        unique_words[word] = 1

len(unique_words)

6042

In [47]:
sorted(unique_words, key=len)[-5:]

['chocolate-coloured',
 'superiors—behold!”',
 'coolness—frightened',
 'gentleman—something',
 'pocket-handkerchief.']

## Punctuation

In [22]:
def split_line(line):
    return line.replace('-', ' ').split()
    
split_line("coolness—frightened")

['coolness—frightened']

In [23]:
import unicodedata

print(unicodedata.category('A'))
print(unicodedata.category('.'))

Lu
Po


In [24]:
# Store the unique punctuation characters in the file
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

# Join the keys into a string
punctuation = ''. join(punc_marks)
print(punctuation)


.’;,-“”:?—‘!()_


In [25]:
def clean_word(word, punctuation):
    """Strip punctuation from a word"""
    return word.strip(punctuation).lower()
    
clean_word('Behold!?-', punctuation)

'behold'

In [26]:
# Identifying unique words by first striping all the punctuation from line
unique_words2 = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word, punctuation)
        unique_words2[word] = 1

len(unique_words2)

4030

In [27]:
sorted(unique_words2, key=len)[-5:]

['circumscription',
 'unimpressionable',
 'superiors—behold',
 'coolness—frightened',
 'gentleman—something']

## Word Frequencies

In [28]:
# Count the frequency of each unique word
word_counter = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word, punctuation)
        if word not in word_counter:
            word_counter[word] = 1
        else:
            word_counter[word] += 1

In [30]:
# Show the five most frequent words
def second_element(t):
    return t[1]

items = sorted(word_counter.items(), key=second_element, reverse=True)

for word, freq in items[:10]:
    print(freq, word, sep='\t')

1608	the
970	and
940	of
644	to
633	i
625	a
469	was
424	in
376	he
368	that


## Optional Parameters

In [51]:
round(3.141592653589793, ndigits=3)

3.142

In [32]:
def print_most_common(word_counter, num=5):
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(freq, word, sep='\t')

print_most_common(word_counter)

1608	the
970	and
940	of
644	to
633	i


## Dictionary Subtraction


In [60]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

Downloaded words.txt


In [33]:
# Split into a list of strings
word_list = open('words.txt').read().split()

# Store the words as keys in a dictionary
valid_words = {}
for word in word_list:
    valid_words[word] = 1

def subtract(d1, d2):
    """Return a dictionary that contains all the keys from one that are not in the other"""
    result = {}
    for key in d1:
        if key not in d2:
            result[key] = d1[key]
    return result

diff = subtract(word_counter, valid_words)
print_most_common(diff)

633	i
625	a
127	utterson
122	mr
97	hyde


In [36]:
# Find words that are mispelled just once
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

print(len(singletons))
print(singletons[-5:])

230
['brought—no', 'alleviation—but', 'circumscription', 'reindue', 'fearstruck']


## Random numbers

In [71]:
import random
# Initialize the random number generator
random.seed(4)

In [77]:
# Choice
t = [1, 2, 3]
random.choice(t)

3

In [81]:
# Choose a random key from a dictionary
words = list(word_counter)  # make a list of words from the keys
random.choice(words)

'stands'

In [82]:
# Generate a random sequence of words
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

knee toiling gloom myself whipping prisonhouse 

In [91]:
# Add weights to words
weights = word_counter.values()
# print(weights)
random_words = random.choices(words, weights=weights, k=6)
' '.join(random_words)

'wish the that if for disappearance'

## Bigrams

In [ ]:
# Bigram = a sequence of two words
# Trigram = a sequence of three words
# n-gram = you get the picture

In [94]:
# Bigram counter
bigram_counter = {}

# Takes a list of two strings
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[bigram] = 1
    else:
        bigram_counter[bigram] += 1

In [97]:
# Keep track to each pair of words
window = []

def process_word(word):
    window.append(word)

    if len(window) == 2:
        count_bigram(window)
        window.pop(0)

In [99]:
# Loop thru the words in the book and process them one at a time
# Bring all the functions used so far down into one cell.

# Clean up the book
def is_special_line(line):
    return line.strip().startswith('*** ')


def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w')

    for line in reader:
        if is_special_line(line):
            break
    
    for line in reader:
        if is_special_line(line):
            break;
        writer.write(line)

    reader.close()
    writer.close()

    
def split_line(line):
    return line.replace('-', ' ').split()


# Store the unique punctuation characters in the file
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

# Join the keys into a string
punctuation = ''. join(punc_marks)
print(punctuation)


def clean_word(word, punctuation):
    return word.strip(punctuation).lower()
    
clean_word('Behold!?-', punctuation)


for line in open(filename):
    for word in split_line(line):
        word = clean_word(word, punctuation)
        process_word(word)


# Remove unwanted metadata text from the file. Write to a new file.
filename = 'dr_jekyll.txt'  
clean_file('pg43.txt', filename)


TypeError: unhashable type: 'list'